<a href="https://colab.research.google.com/github/ayeza527/tmdb_project/blob/main/TMDB_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import requests
from google.colab import userdata

API_KEY = userdata.get("tmdb_key")
url = "https://api.themoviedb.org/3/discover/movie"
params = {
    "api_key": API_KEY,
    "primary_release_date.gte": "2016-01-01",
    "primary_release_date.lte": "2016-12-31",
    "sort_by": "vote_count.desc",
    "page": 1,
}
r = requests.get(url, params=params)
data = r.json()

print("Status:", r.status_code)
print("Total pages:", data["total_pages"])
print("Total movies:", data["total_results"])
print("Top movie:", data["results"][0]["title"])

Status: 200
Total pages: 1001
Total movies: 20001
Top movie: Deadpool


This is a test code to test that api key is working and we can access the data, this code requests only 1 year wort of data and 1 page:

In [8]:
import os
SAVE_DIR = "/content/data"
os.makedirs(SAVE_DIR, exist_ok=True)
print(os.path.exists(SAVE_DIR))

True


In [14]:
import os
import time
import requests
import pandas as pd
from datetime import datetime

DIR = "/content/data"
os.makedirs(DIR, exist_ok=True)

url = "https://api.themoviedb.org/3/discover/movie"

start_time = datetime.now()
req_count = 0
all_ids = []

for year in range(2016, 2027):
    year_ids = []
    for page in range(1, 6):
        r = requests.get(url, {
            "api_key": API_KEY,
            "primary_release_date.gte": f"{year}-01-01",
            "primary_release_date.lte": f"{year}-12-31",
            "sort_by": "vote_count.desc",
            "include_adult": "false",
            "page": page,
        })
        req_count += 1
        data = r.json()

        year_ids += [movie["id"] for movie in data["results"]]
        time.sleep(0.1)
        # we are doing this so alot of request do not at the same time and there is a gap in between to avoid error

    print(year, len(year_ids))
    all_ids += year_ids

all_ids = list(dict.fromkeys(all_ids))
print("Unique IDs:", len(all_ids))
print("Total requests: ", req_count)
pd.Series(all_ids, name="id").to_csv(f"{DIR}/movie_ids.csv", index=False)

2016 100
2017 100
2018 100
2019 100
2020 100
2021 100
2022 100
2023 100
2024 100
2025 100
2026 100
Unique IDs: 1100
Total requests:  55


I used the TMDB discover movies endpoints to collect movie IDs for years 2016 tp 2026. For each year, i requested 5 pages of result to get a total of 100 movies(20 movies per page). I , then removed duplicates by toring itn a dictionary then saved it as a list in movie_ids.csv. Discover filters by release date but can extract little informaton at a time, so i got ids first so i can use them to extract other information in the next step. I also retrieved one year at a time to get an even spread across years, sorted by vote_count descending becase well known films are more likely to have a real budget and revenue values.added a short pause between requests to stay within TMDB’s rate limits, and I counted every request for the collection log.

In [15]:
import json
import hashlib

raw_movies = []
failed_ids = []

for i, movie_id in enumerate(all_ids):
    details_url = f"https://api.themoviedb.org/3/movie/{movie_id}"
    params = {"api_key": API_KEY, "append_to_response": "credits"}

    for attempt in range(3):
        r = requests.get(details_url, params=params)
        request_count += 1
        if r.status_code == 200:
            break
        time.sleep(2)

    if r.status_code == 200:
        movie = r.json()
        movie["collected_at"] = datetime.now().isoformat(timespec="seconds")
        raw_movies.append(movie)
    else:
        failed_ids.append(movie_id)

    if (i + 1) % 100 == 0:
        print("Done:", i + 1, "of", len(all_ids))
    time.sleep(0.05)

end_time = datetime.now()

raw_path = f"{DIR}/tmdb_raw.json"
with open(raw_path, "w") as f:
    json.dump(raw_movies, f)

with open(raw_path, "rb") as f:
    sha256 = hashlib.sha256(f.read()).hexdigest()

print("Movies collected:", len(raw_movies))
print("Failed IDs:", len(failed_ids))
print("Total requests so far:", request_count)
print("SHA-256:", sha256)

Done: 100 of 1100
Done: 200 of 1100
Done: 300 of 1100
Done: 400 of 1100
Done: 500 of 1100
Done: 600 of 1100
Done: 700 of 1100
Done: 800 of 1100
Done: 900 of 1100
Done: 1000 of 1100
Done: 1100 of 1100
Movies collected: 1100
Failed IDs: 0
Total requests so far: 1100
SHA-256: b2eefe568711c6dfd620750d77516a08ad88484eb93cb24e07b34b5fc0a4f63d


For each of the ~1,100 movie IDs collected in the previous step, I requested the full details from the TMDB Movie Details endpoint, using append_to_response=credits to receive cast and crew in the same reply. I added a collected_at timestamp to every movie, saved all the unmodified replies to tmdb_raw.json, and calculated a SHA-256 fingerprint of that file.
SHA256 is a method that takes any file and turns it into a 64-character code whichis a fingerprint that shows this data was collected byou team at a specific time.I also added a retry (up to 3 attempts) and a short pause between requests so temporary failures and rate limits wouldn't break the collection.

In [18]:
import os, json, hashlib
import pandas as pd

SAVE_DIR = "/content/data"
raw_path = f"{SAVE_DIR}/tmdb_raw.json"
print(os.listdir(SAVE_DIR))

with open(raw_path) as f:
    raw_movies = json.load(f)

with open(raw_path, "rb") as f:
    sha256 = hashlib.sha256(f.read()).hexdigest()

print("Movies:", len(raw_movies))
print("SHA-256:", sha256)

['tmdb_raw.json', 'movie_ids.csv']
Movies: 1100
SHA-256: b2eefe568711c6dfd620750d77516a08ad88484eb93cb24e07b34b5fc0a4f63d


In [19]:
stamps = sorted(m["collected_at"] for m in raw_movies)
start_time, end_time = stamps[0], stamps[-1]
print(start_time, end_time)


2026-11-23T16:03:33 2026-11-23T16:06:31


After the Colab session disconnected, the variables stored in memory were lost, so I reloaded the saved raw file tmdb_raw.json into raw_movies and recalculated its SHA-256 fingerprint, instead of collecting the data again, which would have created new timestamps and a different fingerprint. The output shows that the folder contains tmdb_raw.json and movie_ids.csv, that the file holds 1,100 movies, and that the recalculated SHA-256 is identical to the one printed at the end of the collection step. This confirms that the raw data was not modified, so the collection log from before the disconnect is still valid.he disconnect also erased the start_time and end_time variables that the data collection log requires, so I recovered them by sorting the collected_at column of the raw data and taking the earliest and latest values.

In [21]:
import pandas as pd

rows = []
for d in raw_movies:
    credits = d.get("credits", {})
    crew = credits.get("crew", [])
    cast = credits.get("cast", [])
    directors = [c["name"] for c in crew if c.get("job") == "Director"]

    rows.append({
        "id": d["id"],
        "title": d.get("title"),
        "release_date": d.get("release_date"),
        "budget": d.get("budget"),
        "revenue": d.get("revenue"),
        "runtime": d.get("runtime"),
        "popularity": d.get("popularity"),
        "vote_average": d.get("vote_average"),
        "vote_count": d.get("vote_count"),
        "original_language": d.get("original_language"),
        "status": d.get("status"),
        "adult": d.get("adult"),
        "in_collection": d.get("belongs_to_collection") is not None,
        "genres": "|".join(g["name"] for g in d.get("genres", [])),
        "production_companies": "|".join(c["name"] for c in d.get("production_companies", [])),
        "production_countries": "|".join(c["iso_3166_1"] for c in d.get("production_countries", [])),
        "spoken_languages ": len(d.get("spoken_languages", [])),
        "director": directors[0] if directors else None,
        "top_cast": "|".join(c["name"] for c in cast[:3]),
        "cast_size": len(cast),
        "overview": d.get("overview"),
        "tagline": d.get("tagline"),
        "collected_at": d["collected_at"],
    })

df = pd.DataFrame(rows)
df.to_csv(f"{DIR}/tmdb_movies.csv", index=False)

discover_requests = 55
details_requests = 1100

print(" DATA COLLECTION LOG")
print("Roll numbers    : 25L-2571,25L-0900")
print("Source          : https://api.themoviedb.org/3/discover/movie and /movie/{id}?append_to_response=credits")
print("Start           :", start_time)
print("End             :", end_time)
print("Requests        :", discover_requests, "Discover +", details_requests, "Details =", discover_requests + details_requests)
print("Rows collected  :", len(df))
print("collected_at    : column added to every row")
print("SHA-256 (raw)   :", sha256)
df.head()

 DATA COLLECTION LOG
Roll numbers    : 25L-2571,25L-0900
Source          : https://api.themoviedb.org/3/discover/movie and /movie/{id}?append_to_response=credits
Start           : 2026-11-23T16:03:33
End             : 2026-11-23T16:06:31
Requests        : 55 Discover + 1100 Details = 1155
Rows collected  : 1100
collected_at    : column added to every row
SHA-256 (raw)   : b2eefe568711c6dfd620750d77516a08ad88484eb93cb24e07b34b5fc0a4f63d


,id,title,release_date,budget,revenue,runtime,popularity,vote_average,vote_count,original_language,...,genres,production_companies,production_countries,spoken_languages,director,top_cast,cast_size,overview,tagline,collected_at
0,293660,Deadpool,2016-02-09,58000000,782837347,108,21.4464,7.600,33670,en,...,Action|Adventure|Comedy,20th Century Fox|The Donners' Company|Genre Fi...,US,1,Tim Miller,Ryan Reynolds|Morena Baccarin|Ed Skrein,52,The origin story of former Special Forces oper...,Feel the love.,2026-11-23T16:03:33
1,271110,Captain America: Civil War,2016-04-27,250000000,1155046416,147,25.9850,7.456,24905,en,...,Adventure|Action|Science Fiction,Marvel Studios|Vita-Ray Dutch Productions III|...,US|DE,4,Joe Russo,Chris Evans|Robert Downey Jr.|Scarlett Johansson,104,"Following the events of Age of Ultron, the col...",United we stand. Divided we fall.,2026-11-23T16:03:33
2,284052,Doctor Strange,2016-10-25,180000000,676343174,115,17.4811,7.424,24248,en,...,Fantasy|Adventure|Action,Marvel Studios,US,1,Scott Derrickson,Benedict Cumberbatch|Chiwetel Ejiofor|Rachel M...,44,"After his career is destroyed, a brilliant but...",The impossibilities are endless.,2026-11-23T16:03:33
3,297761,Suicide Squad,2016-08-03,175000000,749200054,122,11.9384,5.916,22654,en,...,Action|Adventure|Fantasy,Warner Bros. Pictures|DC Films|Atlas Entertain...,US,3,David Ayer,Will Smith|Jared Leto|Margot Robbie,106,"From DC Comics comes the Suicide Squad, an ant...",Worst. Heroes. Ever.,2026-11-23T16:03:33
4,329865,Arrival,2016-11-10,47000000,203388186,116,22.4036,7.638,20230,en,...,Drama|Science Fiction|Mystery,FilmNation Entertainment|Lava Bear Films|21 La...,US,3,Denis Villeneuve,Amy Adams|Jeremy Renner|Forest Whitaker,72,Taking place after alien crafts land around th...,Why are they here?,2026-11-23T16:03:33


I converted the raw API replies in raw_movies into a structured dataset with one movie per row, and saved it as tmdb_movies.csv. I did this because the raw data is nested JSON (lists and dictionaries inside each movie) which cannot be analysed directly, while the project requires a flat table. From each movie I extracted numerical attributes (budget, revenue, runtime, popularity, vote_average, vote_count, cast_size), categorical attributes (original_language, status, director), multi-valued categorical attributes (genres, production_companies, production_countries, top_cast), Boolean attributes (adult, in_collection), a date attribute (release_date), free-text fields (overview, tagline), and the collected_at timestamp. Multi-valued fields were joined with “|” inside each movie’s own row so that the dataset keeps exactly one row per movie, and the director was picked out of the crew list by filtering for the job “Director”. The data collection log shows that the collection ran from 2026-11-23T16:03:33 to 16:06:31 (about three minutes, in Colab’s server time), using 55 Discover requests and 1,100 Details requests (1,155 in total), and produced 1,100 rows and 23 columns. The SHA-256 fingerprint of the raw file is b2eefe56...f63d. The first five rows show well-known, highly voted films with real budget and revenue values, which is expected because I sorted by vote count. This means the dataset leans towards popular movies, which I will discuss as a limitation, and I still need to check the full dataset for zero budget or revenue values that TMDB uses to mean “unknown”.